[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/07_logistic_regression/01_logistic_regression.ipynb)

# Bài 01 — Logistic Regression

Ở chương trước, đầu ra của bài toán hồi quy là dữ liệu **liên tục**. Trong chương này, ta tìm hiểu bài toán học có giám sát **Logistic Regression** (hồi quy logistic) với đầu ra là dữ liệu **rời rạc** — bài toán **phân loại (classification)**. Dù tên có chữ "regression", đây là một thuật toán phân loại.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/07_logistic_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Có thể dùng Linear Regression cho bài toán phân loại không?

Giả sử cần phân loại một khối u là ác tính (1) hay lành tính (0) dựa trên kích thước khối u. Linear Regression không đưa ra **xác suất**, mà tìm đường thẳng sao cho tổng bình phương khoảng cách nhỏ nhất:

$$J(\theta) = \frac{1}{2m}\sum_{i=1}^{m}\left(h_\theta(x^{(i)}) - y^{(i)}\right)^2$$

Với dữ liệu "đẹp", có vẻ vẫn phân loại được bằng ngưỡng: $h(x) \ge 0.5$ → ung thư, $h(x) < 0.5$ → không ung thư.

<img src="https://i.stack.imgur.com/VVtRW.png" style="width:28%;height:28%;">

Nhưng thêm vài khối u **rất lớn** (chắc chắn ác tính) thì đường thẳng bị kéo lệch, ngưỡng 0.5 không còn đúng nữa:

<img src="https://i.stack.imgur.com/nEC4H.png" style="width:45%;height:45%;">

Tái hiện bằng code *(bổ sung)*:

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression

size = np.r_[rng.uniform(1, 4, 20), rng.uniform(5, 8, 20)]
label = np.r_[np.zeros(20), np.ones(20)]
size_extra = np.r_[size, [18, 20, 22]]                      # thêm vài khối u rất lớn
label_extra = np.r_[label, [1, 1, 1]]
grid = np.linspace(0, 23, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (xs, ys, title) in zip(axes, [(size, label, "Dữ liệu ban đầu"), (size_extra, label_extra, "Thêm 3 khối u rất lớn")]):
    lin = LinearRegression().fit(xs[:, None], ys)
    log = LogisticRegression(C=100).fit(xs[:, None], ys)
    thr = (0.5 - lin.intercept_) / lin.coef_[0]
    ax.scatter(xs, ys, c=ys, cmap="coolwarm", edgecolor="k", zorder=3)
    ax.plot(grid, lin.predict(grid[:, None]), label=f"Linear Regression (ngưỡng 0.5 tại x = {thr:.1f})")
    ax.plot(grid, log.predict_proba(grid[:, None])[:, 1], label="Logistic Regression: P(y=1|x)")
    ax.axhline(0.5, color="gray", ls=":")
    ax.set_ylim(-0.3, 1.5)
    ax.set_xlabel("kích thước khối u")
    ax.set_title(title)
    ax.legend(fontsize=8, loc="upper left")
plt.show()

Vấn đề của Linear Regression cho phân loại:
1. Ngưỡng quyết định bị **dịch chuyển** bởi các điểm "quá dễ" ở xa — những điểm không hề mơ hồ lại làm hỏng ranh giới.
2. Đầu ra có thể $< 0$ hoặc $> 1$ — không phải xác suất.

Ta không thể chỉnh ngưỡng mỗi khi dữ liệu thay đổi. Thay vào đó, dùng **Logistic Regression** cho phân loại nhị phân (0 hay 1); phân loại nhiều lớp ở bài 05–06.

## 2. Bài toán Logistic Regression

### 2.1 Xây dựng hàm giả thuyết $h_\theta(x)$

Ta muốn $0 \le h_\theta(x) \le 1$ → đưa đầu ra tuyến tính qua một **hàm kích hoạt (activation function)** $g$:

$$h_\theta(x) = g(\theta^\top x), \qquad z = \theta^\top x$$

Một số activation function:

<img src="images/ActivationFunction.png" style="width:55%;height:55%;">

Hàm **sigmoid** được dùng vì bị chặn trong $(0, 1)$ — phù hợp để biểu diễn **xác suất**:

$$g(z) = \sigma(z) = \frac{1}{1 + e^{-z}}, \qquad \sigma'(z) = \sigma(z)\big(1 - \sigma(z)\big)$$

<img src="images/sigmoid_function.png" style="width:40%;height:40%;">

**Diễn giải xác suất:**

$$h_\theta(x) = P(y = 1 \mid x; \theta), \qquad P(y = 0 \mid x; \theta) = 1 - h_\theta(x)$$

Ví dụ: nếu xác suất là 1 bằng 70% thì xác suất là 0 bằng 30%. Dự đoán:
- $h_\theta(x) \ge 0.5$ → $y = 1$;  $h_\theta(x) < 0.5$ → $y = 0$.

<img src="images/chap7-logistic_6.png" style="width:45%;height:45%;">

Thử thay đổi $\theta$ để hiểu rõ hơn về sigmoid tại [desmos](https://www.desmos.com/calculator/bgontvxotm): $\theta$ lớn → sigmoid dốc hơn (model "tự tin" hơn); bias dịch đường cong sang trái/phải.

<img src="images/sigmoid-change.png" style="width:45%;height:45%;">

### 2.2 Odds và log-odds *(bổ sung)*

Đảo ngược sigmoid: nếu $p = \sigma(z)$ thì

$$z = \log\frac{p}{1 - p} = \theta^\top x$$

$\frac{p}{1-p}$ là **odds** (tỉ lệ cược: odds = 3 nghĩa là khả năng xảy ra gấp 3 lần không xảy ra), $\log\frac{p}{1-p}$ là **log-odds (logit)**. Vậy Logistic Regression là một **model tuyến tính cho log-odds**.

**Diễn giải hệ số:** tăng $x_j$ thêm 1 đơn vị → log-odds tăng $\theta_j$ → **odds nhân với $e^{\theta_j}$** (*odds ratio*). Ví dụ $\theta_j = 0.7$: odds tăng gấp $e^{0.7} \approx 2$ lần.

In [ ]:
z = np.linspace(-6, 6, 200)
p = 1 / (1 + np.exp(-z))
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
axes[0].plot(z, p)
axes[0].set_title("p = σ(z)")
axes[1].plot(p, p / (1 - p))
axes[1].set_yscale("log")
axes[1].set_title("odds = p/(1−p)")
axes[2].plot(p, np.log(p / (1 - p)))
axes[2].set_title("log-odds = logit(p) — tuyến tính theo θᵀx")
for ax in axes[1:]:
    ax.set_xlabel("p")
plt.show()

### 2.3 Decision Boundary (đường ranh giới quyết định)

Decision boundary là đường (mặt) phân chia ranh giới giữa các lớp. Vì $g(0) = \frac12$, $g(z) \to 1$ khi $z \to \infty$, $g(z) \to 0$ khi $z \to -\infty$:

$$h_\theta(x) \ge 0.5 \;\Longleftrightarrow\; z \ge 0 \;\Longleftrightarrow\; \theta^\top x \ge 0$$

<img src="images/decision-boundary.png" style="width:45%;height:45%;">

**Ví dụ:** $h_\theta(x) = g(\theta_0 + \theta_1 x_1 + \theta_2 x_2)$ với $\theta^\top = [-3, 1, 1]$. Dự đoán $y = 1$ khi $-3 + x_1 + x_2 \ge 0 \Leftrightarrow x_1 + x_2 \ge 3$. Decision boundary là đường thẳng $x_1 + x_2 = 3$.

<img src="images/decision-boundary-1.png" style="width:18%;height:18%;"> <img src="images/decision-boundary-2.png" style="width:18%;height:18%;">

Với feature tuyến tính, decision boundary của Logistic Regression luôn là **siêu phẳng**. Muốn ranh giới cong, thêm feature phi tuyến ($x_1^2, x_1x_2, \dots$) — bài 03.

## 3. Cost Function — từ Maximum Likelihood

Tập train $\{(x^{(i)}, y^{(i)})\}_{i=1}^m$ với $x \in \mathbb{R}^{n+1}$ ($x_0 = 1$), $y \in \{0, 1\}$.

Gộp hai trường hợp $P(y = 1) = h_\theta(x)$ và $P(y = 0) = 1 - h_\theta(x)$ thành một công thức (phân phối **Bernoulli**):

$$p(y \mid x; \theta) = \big(h_\theta(x)\big)^y\big(1 - h_\theta(x)\big)^{1-y}$$

(thay $y = 1$ được $h_\theta(x)$; thay $y = 0$ được $1 - h_\theta(x)$).

Likelihood của toàn bộ dữ liệu (các mẫu độc lập):

$$\mathcal L(\theta) = \prod_{i=1}^{m}\big(h_\theta(x^{(i)})\big)^{y^{(i)}}\big(1 - h_\theta(x^{(i)})\big)^{1 - y^{(i)}}$$

Lấy log, đổi dấu (để cực tiểu) và lấy trung bình → **binary cross-entropy (log loss)**:

$$J(\theta) = -\frac{1}{m}\sum_{i=1}^{m}\Big[y^{(i)}\log h_\theta(x^{(i)}) + (1 - y^{(i)})\log\big(1 - h_\theta(x^{(i)})\big)\Big]$$

**Dạng vector:** $h = g(X\theta)$, $\quad J(\theta) = -\frac{1}{m}\left[y^\top\log h + (1 - y)^\top\log(1 - h)\right]$

> 📝 *Đính chính so với bản gốc:* số hạng thứ hai là $\log\big(1 - h_\theta(x^{(i)})\big)$ (bản gốc in nhầm $\log h(1 - x^{(i)})$); $x \in \mathbb R^{n+1}$ và $y \in \{0, 1\}$.

**Trực giác:** khi $y = 1$, loss là $-\log h$: dự đoán $h \to 1$ thì loss → 0; dự đoán $h \to 0$ (tự tin mà sai) thì loss → ∞. Cross-entropy **phạt rất nặng** các dự đoán tự tin nhưng sai.

### Vì sao không dùng MSE? *(bổ sung)*

Với sigmoid, MSE theo $\theta$ là hàm **không lồi** (có vùng phẳng, gradient gần 0 khi dự đoán sai hoàn toàn) — gradient descent dễ bị chậm/kẹt. Cross-entropy thì **lồi** → có cực tiểu toàn cục duy nhất.

In [ ]:
x1 = np.array([-2.0, -1.0, 1.0, 2.0])
y1 = np.array([1, 0, 1, 0])                   # dữ liệu nhỏ, cố tình "khó"
ws = np.linspace(-10, 10, 400)
mse_loss = [np.mean((1 / (1 + np.exp(-w * x1)) - y1) ** 2) for w in ws]
ce_loss = [-np.mean(y1 * np.log(1 / (1 + np.exp(-w * x1)) + 1e-12) + (1 - y1) * np.log(1 - 1 / (1 + np.exp(-w * x1)) + 1e-12)) for w in ws]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(ws, mse_loss)
axes[0].set_title("MSE theo w: có vùng phẳng, không lồi")
axes[1].plot(ws, ce_loss)
axes[1].set_title("Cross-entropy theo w: lồi")
for ax in axes:
    ax.set_xlabel("w")
plt.show()

## 4. Gradient Descent

Kết quả đạo hàm có dạng **giống hệt** Linear Regression (chỉ khác $h_\theta$ là sigmoid):

$$\theta_j := \theta_j - \alpha\frac{1}{m}\sum_{i=1}^m\left(h_\theta(x^{(i)}) - y^{(i)}\right)x_j^{(i)}, \qquad j = 0, \dots, n$$

**Dạng vector:**

$$\theta := \theta - \frac{\alpha}{m}X^\top\left(g(X\theta) - \vec y\right)$$

**Chứng minh** (tham khảo) — dùng chain rule và $\sigma'(z) = \sigma(z)(1 - \sigma(z))$ (chương 02, bài 04):

<img src="images/gd.png" style="width:60%;height:60%;">

Gradient "sai số × đầu vào" $(h - y)x$ — đơn giản và đẹp, nhờ sigmoid + cross-entropy "triệt tiêu" lẫn nhau. Cùng cấu trúc xuất hiện lại ở softmax (bài 06) và lớp output của neural network (chương 27).

## 5. Advanced Optimization

Ngoài gradient descent, có nhiều thuật toán tối ưu khác giúp tìm $\theta$ nhanh hơn: [Conjugate gradient](https://en.wikipedia.org/wiki/Conjugate_gradient_method), [BFGS](https://en.wikipedia.org/wiki/Broyden%E2%80%93Fletcher%E2%80%93Goldfarb%E2%80%93Shanno_algorithm), [L-BFGS](https://en.wikipedia.org/wiki/Limited-memory_BFGS), Newton's method...

- ✅ Không cần chọn learning rate (có line search bên trong), thường hội tụ nhanh hơn GD.
- ✅ Dễ dùng qua thư viện: `scipy.optimize.minimize`; scikit-learn mặc định dùng **L-BFGS**.
- ⚠️ Không nên tự cài đặt cho dự án thật; khó debug hơn GD.

Bài 02 cài đặt Logistic Regression from scratch bằng GD và so sánh với `scipy.optimize.minimize` (BFGS).

## 6. Tổng kết

| | Công thức |
|---|---|
| Hàm giả thuyết | $h_\theta(x) = g(\theta^\top x)$, $g(z) = \frac{1}{1 + e^{-z}}$ |
| Xác suất | $h_\theta(x) = P(y = 1 \mid x)$; log-odds $= \theta^\top x$ |
| Decision boundary | $\theta^\top x = 0$ |
| Cost function | $J(\theta) = -\frac{1}{m}\left[y^\top\log h + (1 - y)^\top\log(1 - h)\right]$ |
| Gradient descent | $\theta := \theta - \frac{\alpha}{m}X^\top(g(X\theta) - \vec y)$ |

## 🧠 Kiểm tra nhanh

1. Vì sao Linear Regression không phù hợp cho phân loại? Nêu 2 lý do.
2. $\theta^\top = [-3, 1, 1]$, điểm $x = (1, 1)$ được dự đoán lớp nào? Xác suất là bao nhiêu?
3. Hệ số $\theta_j = \ln 2$. Khi $x_j$ tăng 1 đơn vị, odds thay đổi thế nào?
4. Vì sao cross-entropy phù hợp hơn MSE cho Logistic Regression?

## Tài liệu tham khảo

[1] [CS229 — Machine Learning](https://www.coursera.org/learn/machine-learning/resources/Zi29t) · [2] [Linear vs. Logistic Regression on Classification Problems](https://stats.stackexchange.com/questions/22381/why-not-approach-classification-through-regression) · [3] [Logistic Regression — Machine Learning Cơ Bản](https://machinelearningcoban.com/2017/01/27/logisticregression/)

---
<sub>Bài gốc: *Logistic Regression* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 4 (MIT License). Đã đính chính công thức cost function và miền giá trị; bổ sung: tái hiện vấn đề của Linear Regression bằng code, odds/log-odds & diễn giải hệ số, vì sao không dùng MSE.</sub>